# 💡 AI for IoT – Practical 2: Optimization and Deployment Readiness

**Goal:** package the preprocessing and the models of Practical 1 into scikit-learn **pipelines**, measure their **memory** and **time** footprint, and decide whether they can run on an **ESP32** microcontroller.

- **Task 1:** LR-Pipeline and XGB-Pipeline (`StandardScaler` + model), with Accuracy and F1-score
- **Task 2.1:** model size (serialised with `pickle`, in KB)
- **Task 2.2:** inference time (whole test set and per sample)
- **Task 2.3:** critical deployment analysis for the ESP32
- **Bonus:** how far XGBoost can be compressed for the device

**Dataset:** same as Practical 1 ([Kaggle – Smoke Detection Dataset](https://www.kaggle.com/datasets/deepcontractor/smoke-detection-dataset)), with the same features and the same stratified 80/20 split (`random_state = 42`), so the results can be compared directly.

## 1. Setup – Import libraries

In [1]:
# Standard library
import os
import sys
import time
import pickle
import platform

# Data handling
import numpy as np
import pandas as pd

# Machine learning
import sklearn
import xgboost
from sklearn.model_selection import train_test_split, LeaveOneGroupOut
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn import metrics
from xgboost import XGBClassifier
from threadpoolctl import threadpool_limits   # limits NumPy/OpenMP threads during the timings

import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42  # same seed as Practical 1

# Timings depend on the machine, so record where they were measured
print("Python :", platform.python_version())
print("sklearn:", sklearn.__version__, "| xgboost:", xgboost.__version__)
print("CPU    :", platform.processor() or platform.machine(), "|", os.cpu_count(), "logical cores")

Python : 3.14.7
sklearn: 1.9.1 | xgboost: 3.4.1
CPU    : Intel64 Family 6 Model 140 Stepping 1, GenuineIntel | 8 logical cores


## 2. Data – same preparation as Practical 1

- `Unnamed: 0` (row index), `UTC` (timestamp) and `CNT` (session counter) are removed: they are not sensor measurements, and a real detector must decide from what its sensors measure (see Practical 1).
- The 12 remaining sensors are the features; `Fire Alarm` is the target.
- Stratified 80/20 split with `random_state = 42`.

In [2]:
# Look for the CSV next to this notebook, then in the Practical 1 folder, then in the Kaggle inputs
csv_path = None
for root in [".", "../TP1", "/kaggle/input"]:
    for dirpath, _, files in os.walk(root):
        for f in files:
            if f.lower() == "smoke_detection_iot.csv":
                csv_path = os.path.join(dirpath, f)
                break
        if csv_path:
            break
    if csv_path:
        break

assert csv_path is not None, "smoke_detection_iot.csv not found - put it next to this notebook."
df = pd.read_csv(csv_path)
print("Loading:", csv_path, "| shape:", df.shape, "| missing values:", df.isnull().sum().sum())

TARGET = "Fire Alarm"
DROP_COLS = ["Unnamed: 0", "UTC", "CNT"]

X = df.drop(columns=[TARGET] + DROP_COLS)
y = df[TARGET]

# XGBoost forbids '[', ']' and '<' in feature names -> "Temperature[C]" becomes "Temperature_C"
X.columns = X.columns.str.replace("[", "_", regex=False).str.replace("]", "", regex=False)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)
print("Features:", list(X.columns))
print("Train:", X_train.shape, "| Test:", X_test.shape)

Loading: .\smoke_detection_iot.csv | shape: (62630, 16) | missing values: 0


Features: ['Temperature_C', 'Humidity_%', 'TVOC_ppb', 'eCO2_ppm', 'Raw H2', 'Raw Ethanol', 'Pressure_hPa', 'PM1.0', 'PM2.5', 'NC0.5', 'NC1.0', 'NC2.5']
Train: (50104, 12) | Test: (12526, 12)


---
# Task 1 – Pipeline construction

In Practical 1 the scaler and the model were two separate objects: `scaler.fit_transform(X_train)`, then `scaler.transform(X_test)`, then `model.predict(...)`.
Forgetting the scaler at inference time, or fitting it on the test data, silently breaks the model.

A `Pipeline` chains the steps into **one object**:
- `pipeline.fit(X_train, y_train)` fits the scaler on the **training data only**, then trains the model on the scaled data, so no test information leaks into training;
- `pipeline.predict(X_new)` applies the **same stored** mean and standard deviation before predicting;
- the whole chain is saved and deployed as **a single file** (Task 2.1).

In [3]:
def train(pipe):
    # Fit the whole pipeline on the training set and return the training time in seconds
    t0 = time.perf_counter()
    pipe.fit(X_train, y_train)
    return time.perf_counter() - t0

def evaluate(name, pipe):
    # Accuracy and F1-score of a fitted pipeline on the test set
    y_pred = pipe.predict(X_test)
    res = {
        "Model": name,
        "Accuracy": pipe.score(X_test, y_test),   # Pipeline.score() = accuracy for a classifier
        "F1-Score": metrics.f1_score(y_test, y_pred),
    }
    print(f"{name}: Accuracy = {res['Accuracy']:.4f} | F1-Score = {res['F1-Score']:.4f}")
    return res

## Task 1.1 – Logistic Regression pipeline (LR-Pipeline)
`StandardScaler` → `LogisticRegression`, with the same settings as Practical 1.

In [4]:
lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

lr_train_time = train(lr_pipeline)
res_lr = evaluate("LR-Pipeline", lr_pipeline)
print(f"Training time: {lr_train_time:.3f} s")
lr_pipeline

LR-Pipeline: Accuracy = 0.8951 | F1-Score = 0.9281
Training time: 0.306 s


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](12,)","['Temperature_C','Humidity_%','TVOC_ppb',...,'NC0.5','NC1.0','NC2.5']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,12
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


## Task 1.2 – XGBoost pipeline (XGB-Pipeline)
`StandardScaler` → `XGBClassifier`, with the same hyperparameters as Practical 1 (300 trees, depth 6).

Trees do not need scaling: a split such as `Temperature_C < 25` simply becomes `Temperature_C_scaled < (25 − mean) / std`, so the scaler does not change what the trees can learn.
It is kept for **MLOps standardisation**: both models now have the same interface (raw sensor values in, prediction out) and can be swapped without changing the rest of the code.
On the device it costs only 12 subtractions and 12 multiplications per sample.

In [5]:
xgb_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])

xgb_train_time = train(xgb_pipeline)
res_xgb = evaluate("XGB-Pipeline", xgb_pipeline)
print(f"Training time: {xgb_train_time:.3f} s")

XGB-Pipeline: Accuracy = 1.0000 | F1-Score = 1.0000
Training time: 2.352 s


### Task 1 – Results

In [6]:
pipelines = {"LR-Pipeline": lr_pipeline, "XGB-Pipeline": xgb_pipeline}

task1 = pd.DataFrame([res_lr, res_xgb]).set_index("Model")
task1["Training time (s)"] = [lr_train_time, xgb_train_time]
task1.round(4)

,Accuracy,F1-Score,Training time (s)
Model,,,
LR-Pipeline,0.8951,0.9281,0.3057
XGB-Pipeline,1.0000,1.0000,2.3516


The scores are **identical to Practical 1** (LR: Accuracy 0.8951, F1 0.9281; XGBoost: 1.0000): the pipeline performs exactly the same computation as the separate scaler + model, but as one object.

⚠️ Reminder from Practical 1: XGBoost's 1.0000 is inflated by **temporal leakage** (consecutive, almost identical samples end up in both sets).
The deeper analysis of Practical 1 also showed that the dataset holds only **3 independent recordings**: two sessions are twins of two others, with the same values except an artificial temperature.
On a recording they have never seen, both models miss almost every fire (Recall **0.0037** for Logistic Regression and **0.0077** for XGBoost).
These test scores therefore describe data close to the training data. This practical compares the two models on **efficiency**, which does not depend on this problem.

---
# Task 2.1 – Model size (memory footprint)

Each fitted pipeline (scaler + model) is serialised with `pickle`.
The size is measured on the file (`os.path.getsize`) and, as a cross-check, with `sys.getsizeof` on the pickled bytes (it adds the ~33-byte header of a Python `bytes` object).

In [7]:
os.makedirs("models", exist_ok=True)

size_rows = []
for name, pipe in pipelines.items():
    path = os.path.join("models", f"{name}.pkl")
    with open(path, "wb") as f:
        pickle.dump(pipe, f)
    size_rows.append({
        "Model": name,
        "Size (KB)": os.path.getsize(path) / 1024,                        # file on disk
        "sys.getsizeof (KB)": sys.getsizeof(pickle.dumps(pipe)) / 1024,   # same bytes held in memory
    })

sizes = pd.DataFrame(size_rows).set_index("Model")
ratio = sizes.loc["XGB-Pipeline", "Size (KB)"] / sizes.loc["LR-Pipeline", "Size (KB)"]
print(f"XGB-Pipeline is {ratio:.0f} times larger than LR-Pipeline")
sizes.round(2)

XGB-Pipeline is 224 times larger than LR-Pipeline


,Size (KB),sys.getsizeof (KB)
Model,,
LR-Pipeline,1.64,1.67
XGB-Pipeline,367.11,367.15


In [8]:
# Reload check: each saved file gives exactly the same predictions as the trained pipeline
for name, pipe in pipelines.items():
    with open(os.path.join("models", f"{name}.pkl"), "rb") as f:
        reloaded = pickle.load(f)
    same = np.array_equal(reloaded.predict(X_test), pipe.predict(X_test))
    print(f"{name}: reloaded pipeline gives identical predictions -> {same}")

LR-Pipeline: reloaded pipeline gives identical predictions -> True


XGB-Pipeline: reloaded pipeline gives identical predictions -> True


### What does the ESP32 really have to store?
A pickle file contains Python objects: class names, metadata, and for XGBoost extra statistics for every node (gain, cover).
An ESP32 does not run Python with scikit-learn, so on the device the model is exported as **plain C arrays** (for example with `emlearn`, `m2cgen` or `micromlgen`).
Counting only the numbers that a prediction actually uses gives the real on-device footprint:
- **LR-Pipeline:** scaler mean and std (12 + 12) + one weight per feature (12) + bias (1), stored as `float32`;
- **XGB-Pipeline:** the scaler (24 floats) + every node of every tree. We count **12 bytes per node** (`float32` threshold or leaf value, two `uint16` child indices, one `uint8` feature index, padded).

In [9]:
BYTES_FLOAT32 = 4
BYTES_PER_NODE = 12
n_features = X.shape[1]

def c_array_bytes(pipe):
    # Estimated size of the model exported as C arrays (only the numbers used by predict)
    scaler_bytes = 2 * n_features * BYTES_FLOAT32
    model = pipe.named_steps["model"]
    if isinstance(model, LogisticRegression):
        return scaler_bytes + (model.coef_.size + model.intercept_.size) * BYTES_FLOAT32
    return scaler_bytes + len(model.get_booster().trees_to_dataframe()) * BYTES_PER_NODE

booster = xgb_pipeline.named_steps["model"].get_booster()
trees = booster.trees_to_dataframe()
n_trees = trees["Tree"].nunique()
n_leaves = (trees["Feature"] == "Leaf").sum()

print(f"LR : {2 * n_features + n_features + 1} float32 numbers")
print(f"XGB: {n_trees} trees, {len(trees)} nodes ({len(trees) - n_leaves} splits + {n_leaves} leaves)")
print(f"XGB native binary model (booster.save_raw('ubj')): {len(booster.save_raw('ubj')) / 1024:.1f} KB")

SRAM_KB, FLASH_KB = 520, 4 * 1024   # typical ESP32-WROOM-32: 520 KB SRAM, 4 MB flash

footprint = pd.DataFrame({
    "Pickle file (KB)": sizes["Size (KB)"],
    "C arrays, estimate (KB)": [c_array_bytes(p) / 1024 for p in pipelines.values()],
}, index=sizes.index)
footprint["Pickle, % of 520 KB SRAM"] = 100 * footprint["Pickle file (KB)"] / SRAM_KB
footprint["C arrays, % of 520 KB SRAM"] = 100 * footprint["C arrays, estimate (KB)"] / SRAM_KB
footprint["Pickle, % of 4 MB Flash"] = 100 * footprint["Pickle file (KB)"] / FLASH_KB
footprint.round(2)

LR : 37 float32 numbers
XGB: 300 trees, 4922 nodes (2311 splits + 2611 leaves)
XGB native binary model (booster.save_raw('ubj')): 357.2 KB


,Pickle file (KB),"C arrays, estimate (KB)","Pickle, % of 520 KB SRAM","C arrays, % of 520 KB SRAM","Pickle, % of 4 MB Flash"
Model,,,,,
LR-Pipeline,1.64,0.14,0.31,0.03,0.04
XGB-Pipeline,367.11,57.77,70.60,11.11,8.96


---
# Task 2.2 – Computational time

Measurement protocol:
- **One CPU thread** for both models (`n_jobs=1` for XGBoost, `threadpool_limits(1)` for NumPy). The ESP32 has only 2 cores and no thread pool, so a multi-threaded laptop measurement would favour XGBoost unfairly.
- One warm-up call, then the **median of 20 runs** of `.predict(X_test)`. A single run is noisy (other programs, CPU frequency changes).
- **Single inference time = total time / number of test samples.**

In [10]:
xgb_pipeline.set_params(model__n_jobs=1)   # predict on one thread, like Logistic Regression

def time_predict(pipe, X_eval, repeats=20):
    # Median wall-clock time (s) of pipe.predict(X_eval), after one warm-up call
    pipe.predict(X_eval)
    runs = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        pipe.predict(X_eval)
        runs.append(time.perf_counter() - t0)
    return float(np.median(runs))

n_test = len(X_test)
time_rows = []
with threadpool_limits(limits=1):
    for name, pipe in pipelines.items():
        total_s = time_predict(pipe, X_test)
        time_rows.append({
            "Model": name,
            "Total Test Inference Time (s)": total_s,
            "Single Inference Time (ms)": total_s / n_test * 1000,
        })

timing = pd.DataFrame(time_rows).set_index("Model")
print(f"Test set: {n_test} samples")
timing.round(6)

Test set: 12526 samples


,Total Test Inference Time (s),Single Inference Time (ms)
Model,,
LR-Pipeline,0.006844,0.000546
XGB-Pipeline,0.116493,0.009300


### Extra: latency of one real prediction call
Dividing the batch time by 12,526 gives the **amortised** cost per sample: NumPy and XGBoost process the whole test set in one vectorised call.
In the real application the sensor produces **one sample per second**, so each `.predict()` call receives a single row and pays the full Python/library overhead every time.
We measure this case too, on 200 different test samples.

In [11]:
single_rows = [X_test.iloc[[i]] for i in range(200)]   # 200 one-sample inputs, prepared before timing

def time_one_call(pipe, rows):
    # Median time (ms) of one predict() call on a single sample
    pipe.predict(rows[0])
    runs = []
    for r in rows:
        t0 = time.perf_counter()
        pipe.predict(r)
        runs.append(time.perf_counter() - t0)
    return float(np.median(runs)) * 1000

with threadpool_limits(limits=1):
    timing["One-sample call (ms)"] = [time_one_call(p, single_rows) for p in pipelines.values()]

timing["One-sample call, % of the 1 s budget"] = timing["One-sample call (ms)"] / 1000 * 100
timing.round(5)

,Total Test Inference Time (s),Single Inference Time (ms),One-sample call (ms),"One-sample call, % of the 1 s budget"
Model,,,,
LR-Pipeline,0.00684,0.00055,3.1974,0.31974
XGB-Pipeline,0.11649,0.00930,5.0557,0.50557


### Order-of-magnitude estimate on the ESP32
The numbers above come from a laptop CPU (GHz clock, SIMD units, large caches) running Python.
The ESP32 runs at **240 MHz** and executes the model as compiled C, without Python.
Counting the operations of one prediction gives a rough estimate (**an estimate, not a measurement**):
- **LR:** scaler (12 subtractions + 12 multiplications by the pre-computed 1/std) + 12 multiply-adds + 1 comparison of the logit with 0 (the same as P(fire) ≥ 0.5, so no exponential is needed);
- **XGBoost:** scaler (24 operations), then in each tree one comparison per level from the root to a leaf (at most `max_depth` = 6) and one addition of the leaf value.

The cost of one operation, including memory access, is taken between **20 cycles** (data in cache) and **200 cycles** (each access misses the flash cache).

In [12]:
CPU_HZ = 240e6   # ESP32 clock: 240 MHz
max_depth = xgb_pipeline.named_steps["model"].max_depth

esp32 = pd.DataFrame({
    "Operations per prediction": [
        2 * n_features + n_features + 1,                 # LR: scaler + dot product + sign test
        2 * n_features + n_trees * (max_depth + 1),      # XGB: scaler + worst-case path + leaf addition, in every tree
    ],
}, index=list(pipelines))
for cycles in (20, 200):
    esp32[f"ESP32 estimate, {cycles} cycles/op (ms)"] = esp32["Operations per prediction"] * cycles / CPU_HZ * 1000
esp32["Real-time budget (ms)"] = 1000
esp32.round(4)

,Operations per prediction,"ESP32 estimate, 20 cycles/op (ms)","ESP32 estimate, 200 cycles/op (ms)",Real-time budget (ms)
LR-Pipeline,37,0.0031,0.0308,1000
XGB-Pipeline,2124,0.1770,1.7700,1000


---
# 🏆 Bonus – How far can XGBoost be compressed?

The size of a tree ensemble grows with the **number of trees** and with **2^depth** (the number of nodes per tree).
We retrain smaller XGB-Pipelines and measure size, latency and quality.
Quality is measured in two ways:
- **F1 on the random split**: inflated by temporal leakage (see Practical 1), almost every model gets ≈ 1.0;
- **Recall on unseen recordings** (leave one recording out, as in Practical 1, D.5): the two sessions of a twin pair stay together, and each of the 3 recordings is predicted by a model that never saw it.

With fewer trees, each tree must make bigger corrections, so the learning rate is increased for the small models.

In [13]:
# Recording sessions: the CNT counter restarts at 0 at the beginning of each session (see Practical 1)
session = (df["CNT"].diff() < 0).cumsum()

# Twin sessions (Practical 1, D.4): same length, same labels, same values for every sensor except the temperature
no_temp = X.columns.drop("Temperature_C")
twins = {}
ids = sorted(session.unique())
for i, a in enumerate(ids):
    for b in ids[i + 1:]:
        A, B = session == a, session == b
        if A.sum() == B.sum() and (X.loc[A, no_temp].values == X.loc[B, no_temp].values).all() \
                and (y[A].values == y[B].values).all():
            twins[b] = a
recording = session.replace(twins)   # both sessions of a twin pair form one recording
print("Twin sessions:", [(int(a), int(b)) for b, a in twins.items()], "->", recording.nunique(), "independent recordings")

def loro_recall(make_pipe):
    # Recall of the out-of-fold predictions, one fold per independent recording
    y_oof = np.zeros(len(y), dtype=int)
    for train_idx, test_idx in LeaveOneGroupOut().split(X, y, groups=recording):
        model = make_pipe().fit(X.iloc[train_idx], y.iloc[train_idx])
        y_oof[test_idx] = model.predict(X.iloc[test_idx])
    return metrics.recall_score(y, y_oof)

def make_xgb(n_estimators, max_depth, learning_rate):
    # Factory: a new, untrained XGB-Pipeline with the given size
    return lambda: Pipeline([
        ("scaler", StandardScaler()),
        ("model", XGBClassifier(
            n_estimators=n_estimators, max_depth=max_depth, learning_rate=learning_rate,
            subsample=0.9, colsample_bytree=0.9, eval_metric="logloss",
            random_state=RANDOM_STATE, n_jobs=-1,
        )),
    ])

make_lr = lambda: Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

candidates = {"LR-Pipeline": make_lr}
for n_est, depth, lr_rate in [(300, 6, 0.1), (100, 4, 0.3), (50, 3, 0.3), (20, 3, 0.3), (10, 2, 0.5)]:
    candidates[f"XGB {n_est} trees, depth {depth}"] = make_xgb(n_est, depth, lr_rate)

# 1) Train every candidate on the training set
fitted = {name: make_pipe().fit(X_train, y_train) for name, make_pipe in candidates.items()}

# 2) Time them all together on one thread, before the heavy multi-threaded training on each recording
with threadpool_limits(limits=1):
    for name, pipe in fitted.items():
        if "XGB" in name:
            pipe.set_params(model__n_jobs=1)
    call_ms = {name: time_one_call(pipe, single_rows) for name, pipe in fitted.items()}

bonus = pd.DataFrame({
    "Pickle (KB)": {n: len(pickle.dumps(p)) / 1024 for n, p in fitted.items()},
    "C arrays, estimate (KB)": {n: c_array_bytes(p) / 1024 for n, p in fitted.items()},
    "One-sample call (ms)": call_ms,
    "F1 (random split)": {n: metrics.f1_score(y_test, p.predict(X_test)) for n, p in fitted.items()},
})
bonus.index.name = "Model"

# 3) Recall on unseen recordings (each candidate is retrained once per recording)
bonus["Recall (unseen recording)"] = [loro_recall(candidates[n]) for n in bonus.index]
bonus.round(4)

Twin sessions: [(0, 1), (3, 4)] -> 3 independent recordings


,Pickle (KB),"C arrays, estimate (KB)",One-sample call (ms),F1 (random split),Recall (unseen recording)
Model,,,,,
LR-Pipeline,1.6367,0.1445,3.5612,0.9281,0.0037
"XGB 300 trees, depth 6",367.1094,57.7734,5.6034,1.0000,0.0077
"XGB 100 trees, depth 4",122.3633,16.8281,5.3276,1.0000,0.0078
"XGB 50 trees, depth 3",65.8359,8.2031,5.1289,0.9999,0.0004
"XGB 20 trees, depth 3",33.0732,3.4453,5.1712,0.9995,0.0365
"XGB 10 trees, depth 2",19.4512,0.9141,5.1454,0.9973,0.0001


---
# 📝 Results and conclusion

Every number below comes from the outputs of this notebook (`random_state = 42`).
Timings were measured on a laptop CPU (Intel, 8 logical cores, one thread for inference). They depend on the load and the power mode of the machine (a laptop on battery runs slower), so they change between runs; the ratios between the two models stay similar.

### Task 1 – Pipelines (test set: 12,526 samples)
| Model | Accuracy | F1-Score | Training time (s) |
|---|---|---|---|
| LR-Pipeline | 0.8951 | 0.9281 | 0.306 |
| XGB-Pipeline | 1.0000 | 1.0000 | 2.352 |

### Task 2.1 – Model size
| Model | Size (KB) |
|---|---|
| LR-Pipeline | **1.64** |
| XGB-Pipeline | **367.11** |

### Task 2.2 – Computational time
| Model | Total Test Inference Time (s) | Single Inference Time (ms) |
|---|---|---|
| LR-Pipeline | **0.0068** | **0.000546** |
| XGB-Pipeline | **0.1165** | **0.009300** |

Extra measurements:
| Model | C arrays on the device, estimate (KB) | One-sample `predict()` call in Python (ms) | ESP32 estimate per prediction (ms) |
|---|---|---|---|
| LR-Pipeline | 0.14 (37 numbers) | 3.20 | 0.003 – 0.03 |
| XGB-Pipeline | 57.77 (4,922 nodes) | 5.06 | 0.18 – 1.77 |

## Task 2.3 – Critical deployment analysis (the ESP32 question)

**1. Memory feasibility.**
**Flash** is not a problem for either model: the serialised LR-Pipeline takes 1.64 KB (0.04% of a 4 MB flash) and the XGB-Pipeline 367.11 KB (about 9%).
**RAM** is the real constraint, because the model has to be loaded into RAM for prediction.
LR-Pipeline would use only 0.3% of the 520 KB SRAM. XGB-Pipeline would use **70.6%** of the total SRAM.
In practice only part of these 520 KB is free for the application heap (the system, the Wi-Fi/Bluetooth stacks and the sensor buffers take a large share, and the free heap is split into several separate regions), so a 367 KB model **does not realistically fit in RAM**.
**XGBoost is clearly the memory-constrained model: it is 224 times larger than Logistic Regression.**
The pickle file is not what would actually run on the device, though: it stores Python objects and extra statistics for every node.
Exported as C arrays, LR needs only 37 `float32` numbers (148 bytes) and XGBoost about 58 KB (4,922 nodes). Declared `const`, these arrays stay in Flash and are read through the cache, so prediction only needs a few bytes of RAM for the 12 input values.
Logistic Regression is therefore feasible in any form. XGBoost is feasible **only after it is exported to C** (or compressed), not as the raw serialised model.

**2. Time efficiency.**
With one prediction every second, the budget is **1,000 ms per prediction**.
XGBoost is about **17 times slower** than Logistic Regression (0.1165 s vs 0.0068 s for the whole test set), but the measured single inference time is tiny for both: **0.000546 ms** (0.55 µs) for LR and **0.0093 ms** (9.3 µs) for XGBoost, more than 100,000 times below the budget.
Even the realistic case, where `predict()` is called on one sample at a time and the Python overhead dominates, takes only 3.20 ms (LR) and 5.06 ms (XGBoost), i.e. **0.32% and 0.51% of the 1 s budget**.
On the ESP32 (240 MHz, compiled C, no Python) the operation count gives 37 operations for LR (≈ 0.003–0.03 ms) and at most 2,124 for XGBoost (≈ 0.18–1.77 ms). Even if the real cost were 100 times worse than the pessimistic estimate, XGBoost would take about 177 ms, still within the budget.
**Both models are fast enough: time is not the limiting factor, memory is.**
The difference still matters for a battery-powered node: LR needs about 57 times fewer operations per prediction, so the CPU can sleep longer between two readings.

**3. Conclusion.**
**Based purely on efficiency and deployment constraints, Logistic Regression is the clear choice.** It is 224 times smaller (1.64 KB vs 367.11 KB), about 17 times faster (0.000546 ms vs 0.0093 ms per sample), and on the device it is just a dot product over 37 numbers (148 bytes). It needs no special library and fits the ESP32 with a huge margin, even without any optimisation.
Efficiency is not the only criterion for a fire alarm. On the random split XGBoost detects every fire (Recall 1.0000 against 0.9482 for Logistic Regression).
But Practical 1 showed that on a recording they have never seen, **both models miss almost every fire** (Recall 0.0037 and 0.0077), so this dataset cannot show that XGBoost would detect more fires in a new place.
If the higher score of XGBoost is still wanted, the less efficient model can be made deployable with the following optimisation techniques:
- **Fewer and shallower trees** (`n_estimators`, `max_depth`). The bonus experiment shows that **20 trees of depth 3** take 33.07 KB as pickle (11 times smaller) and about 3.45 KB as C arrays (17 times smaller), with at most 104 operations per prediction, and keep F1 = 0.9995 on the random split. With 10 trees of depth 2, F1 drops to 0.9973. On unseen recordings every variant stays below a Recall of 0.04, so they cannot be ranked there.
- **Export to C code** with `emlearn`, `m2cgen` or `micromlgen`. This removes the Python/pickle overhead (367 KB → ≈ 58 KB for the full model), and the model stays in Flash, so it is never copied into RAM.
- **Quantisation**: store thresholds and leaf values as `int16`/`int8` instead of `float32`. This divides the size by 2–4, and integer comparisons are faster on a microcontroller.
- **Fold the scaler into the model**: rewrite the tree thresholds (or the LR weights) in raw sensor units, so the 24 scaler numbers and 24 operations disappear.
- **Feature selection and early stopping**: drop the least important sensors (less to store, read and power), and stop adding trees when the validation loss stops improving.

**Final recommendation:** on efficiency, deploy the **LR-Pipeline**: it is the smallest and fastest model, and it fits the ESP32 without any optimisation.
A compressed XGBoost (≈ 20 trees, depth 3) exported to C would also fit, with a few KB of Flash and well under 1 ms per prediction.
But Practical 1 showed that neither model is ready for a real fire detector yet, because both fail on a recording they have never seen. The first step is to collect training data from more rooms, devices and fire types; the choice of model comes after.

### Limits of this analysis
- The timings were measured on a laptop, not on the ESP32. The ESP32 values are **estimates from operation counts**. A real validation means flashing the exported model and timing it with `esp_timer_get_time()`.
- The C-array size of XGBoost assumes 12 bytes per node. The real size depends on the exporter (code generators such as `m2cgen` produce `if/else` code, which can be larger than arrays).
- 520 KB SRAM and 4 MB Flash correspond to a classic ESP32 (ESP32-WROOM-32). Other variants (for example ESP32-S3 with external PSRAM) have more memory.
- The dataset holds only 3 independent recordings (Practical 1, D.4). On an unseen recording every model misses almost all fires, so this data cannot show which model, or which size of XGBoost, would detect more fires in a new place.